# Predicting Diabetes Risk: A Machine Learning Classification Workflow

**Author:** Sébastien Bodrero
**Programme:** MSc in Artificial Intelligence — Woolf University / Udacity
**Module 3:** Machine Learning Foundations
**Date:** April 2026

---

This notebook implements a complete supervised machine learning workflow applied to the **Pima Indians Diabetes Dataset** (768 rows × 9 columns). The task is **binary classification**: predict whether a patient is likely to have diabetes (Outcome = 1) based on eight diagnostic measurements.

The workflow follows six structured sections:

1. **Setup** — library imports and version reporting
2. **Data Ingestion** — auto-download and initial inspection
3. **Data Preparation & Preprocessing** — handling zero-encoded missing values, scaling, train/test split
4. **Model Selection & Training** — Logistic Regression (baseline) and Random Forest (primary model)
5. **Evaluation** — metrics, confusion matrix, ROC curve, feature importance
6. **Notebook Summary** — findings, challenges, and limitations

## 1. Setup

In [1]:
import urllib.request
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
import sklearn

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print(f'NumPy      {np.__version__}')
print(f'Pandas     {pd.__version__}')
print(f'Matplotlib {matplotlib.__version__}')
print(f'Seaborn    {sns.__version__}')
print(f'scikit-learn {sklearn.__version__}')

NumPy      2.4.4
Pandas     3.0.2
Matplotlib 3.10.8
Seaborn    0.13.2
scikit-learn 1.8.0


## 2. Data Ingestion

The **Pima Indians Diabetes Dataset** was originally compiled by the National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK) and made publicly available through the UCI Machine Learning Repository and Kaggle. It contains diagnostic measurements for 768 female patients of Pima Indian heritage aged 21 and above.

| Column | Type | Description |
|--------|------|-------------|
| `Pregnancies` | int | Number of pregnancies |
| `Glucose` | int | Plasma glucose concentration (2-hour oral glucose tolerance test) |
| `BloodPressure` | int | Diastolic blood pressure (mm Hg) |
| `SkinThickness` | int | Triceps skin fold thickness (mm) |
| `Insulin` | int | 2-hour serum insulin (μU/ml) |
| `BMI` | float | Body mass index (kg/m²) |
| `DiabetesPedigreeFunction` | float | Genetic risk score based on family history |
| `Age` | int | Age in years |
| `Outcome` | int | Target: 1 = diabetes, 0 = no diabetes |

The dataset is downloaded automatically if not already present locally.

In [2]:
DATA_URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
DATA_FILE = 'diabetes.csv'

COLUMN_NAMES = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

if not os.path.exists(DATA_FILE):
    print(f'Downloading {DATA_FILE} ...')
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print('Download complete.')
else:
    print(f'{DATA_FILE} already present — skipping download.')

df = pd.read_csv(DATA_FILE, header=None, names=COLUMN_NAMES)
print(f'\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

Download complete.

Dataset shape: 768 rows × 9 columns


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
